# Analysis and Synthesis of Bell Sounds




In [1]:
import os, sys, wave, struct

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import IPython

from scipy.io.wavfile import write, read as wavread
from scipy.linalg import hankel
import scipy.linalg as la
from matplotlib import cm

print('Imports OK')

: 

---
# 1. Utility Functions

In [ ]:
def read_values(filename):
    """
    Read a WAV file and return:
      T                  : duration in seconds
      data               : 2-D array, shape (nchannels, nframes)
      nframes, nchannels, sampling_frequency
    """
    wave_file = wave.open(filename, 'r')
    nframes   = wave_file.getnframes()
    nchannels = wave_file.getnchannels()
    sampling_frequency = wave_file.getframerate()
    T         = nframes / float(sampling_frequency)
    read_frames = wave_file.readframes(nframes)
    wave_file.close()
    # BUG FIX: original code had operator precedence error:
    #   "%dh" % nchannels*nframes  =>  ("%dh" % nchannels)*nframes (wrong!)
    # Correct: format the total number of shorts
    data = struct.unpack("%dh" % (nchannels * nframes), read_frames)
    data_per_channel = [data[offset::nchannels] for offset in range(nchannels)]
    return T, np.array(data_per_channel), nframes, nchannels, sampling_frequency


def plot_sound(data, times, xlim1, xlim2, name='default_name',
               save=False, w=20, h=5, lw=1):
    """
    Plot a 1-D audio signal.
    BUG FIX: original used undefined globals `x` and `len_x` inside the function.
    Now correctly uses the `data` argument.
    """
    plt.figure(figsize=(w, h))
    plt.plot(times, data.ravel(), linewidth=lw)   # FIX: was x.reshape(len_x)
    plt.xlim(xlim1, xlim2)
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.title(name)
    plt.tight_layout()
    if save:
        plt.savefig(name + '.png', dpi=100)
    plt.show()

In [ ]:
def Synthesis(N, delta, f, a, phi, RSB=False):
    """
    Synthesise an Exponential Sinusoidal Model (ESM) signal:
      s[t] = sum_k  a_k * exp(delta_k*t) * exp(j*(2pi*f_k*t + phi_k))
    Returns the real part for physical signals (or complex for analysis).
    RSB: if a number, adds complex Gaussian noise at that SNR (dB).
    """
    delta = np.asarray(delta, dtype=float)
    f     = np.asarray(f,     dtype=float)
    a     = np.asarray(a,     dtype=float)
    phi   = np.asarray(phi,   dtype=float)

    t     = np.arange(N)
    logz  = delta + 1j * 2 * np.pi * f        # shape (K,)
    alpha = a * np.exp(1j * phi)               # shape (K,)
    # outer product: (K,) x (N,)  =>  (K, N)
    x = np.sum(alpha[:, None] * np.exp(np.outer(logz, t)), axis=0)  # (N,)

    if RSB is False:
        return x
    # Add complex Gaussian noise at given SNR
    Ex = np.real(np.mean(np.abs(x)**2))
    b  = np.random.normal(size=N) + 1j * np.random.normal(size=N)
    Eb = np.real(np.mean(np.abs(b)**2))
    b  = b * np.sqrt(Ex / Eb) * 10**(-RSB / 20)
    return x + b

---
# 2. High-Resolution Core Functions

## 2.1 Empirical Covariance Matrix

In [ ]:
def empirical_autocov(x, n, l):
    """
    Compute the empirical covariance matrix:
        Rxx = (1/l) * X @ X^H
    where X is the n x l Hankel matrix built from signal x of length N = n+l-1.

    BUGS FIXED vs original:
    - original computed hankel wrong: np.hankel(X[:l], X[:n]) swaps n and l roles
      and was not transposed correctly.
    - original never returned the result (missing `return`).
    - now uses scipy.linalg.hankel correctly: first col = x[:n], last row = x[n-1:].
    """
    x  = np.asarray(x)
    N  = len(x)
    assert N == n + l - 1, f'Length mismatch: need N=n+l-1={n+l-1}, got {N}'
    # Build the n×l Hankel matrix
    X   = hankel(x[:n], x[n-1:])   # shape (n, l)
    Rxx = (1.0 / l) * (X @ X.conj().T)
    return Rxx

## 2.2 Signal Subspace Estimation

In [ ]:
def extract_signal_subspace(x, n, l, K):
    """
    Returns W : n×K matrix whose columns form an orthonormal basis of the
    signal subspace, obtained from the K dominant left singular vectors of Rxx.

    BUGS FIXED vs original:
    - n and l were not passed to empirical_autocov (signature mismatch).
    - scipy.linalg.svd returns (U, s, Vh) — singular values, not eigenvalues;
      columns of U already sorted by decreasing singular value.
    """
    Rxx      = empirical_autocov(x, n, l)
    U, s, Vh = la.svd(Rxx)           # U : (n, n), s : (n,), Vh : (n, n)
    W        = U[:, :K]              # n × K signal subspace basis
    W_perp   = U[:, K:]              # n × (n-K) noise subspace basis
    return W, W_perp

## 2.3 ESPRIT — Frequencies and Damping Factors

In [ ]:
def ESPRIT(x, n, K):
    """
    ESPRIT method.
    Parameters
    ----------
    x : 1-D array, length N = n + l - 1
    n : number of rows of the Hankel matrix
    K : number of complex poles to estimate

    Returns
    -------
    delta : (K,) array of damping factors
    f     : (K,) array of normalised frequencies in (-0.5, 0.5]

    BUGS FIXED vs original:
    - W↑ and W↓ were obtained by slicing *columns* instead of *rows*.
      For ESPRIT we need:
         W_down = W[:-1, :]   (remove last ROW)
         W_up   = W[1:,  :]   (remove first ROW)
    - la.eig returns (eigenvalues, eigenvectors); original stored the tuple as Z.
    - l was not defined; now derived from N and n.
    """
    N   = len(x)
    l   = N - n + 1
    W, _ = extract_signal_subspace(x, n, l, K)

    W_down = W[:-1, :]                          # (n-1) × K
    W_up   = W[1:,  :]                          # (n-1) × K

    # Phi = pinv(W_down) @ W_up
    Phi = np.linalg.pinv(W_down) @ W_up         # K × K

    eigenvalues, _ = la.eig(Phi)                # FIX: unpack tuple

    delta = np.log(np.abs(eigenvalues))          # δ_k = ln|z_k|
    f     = np.angle(eigenvalues) / (2 * np.pi) # f_k = angle(z_k) / 2π
    return delta, f

## 2.4 Least Squares — Amplitudes and Phases

In [ ]:
def LeastSquares(x, delta, f):
    """
    Given the signal x, the damping factors delta and frequencies f,
    estimate complex amplitudes alpha_k = a_k * exp(j*phi_k) via
        alpha = pinv(V_N) @ x
    where V_N is the N×K Vandermonde matrix with V_N[t, k] = z_k^t.

    Returns
    -------
    a   : (K,) real amplitudes
    phi : (K,) phases in (-π, π]
    """
    delta = np.asarray(delta, dtype=float)
    f     = np.asarray(f,     dtype=float)
    x     = np.asarray(x)

    N, K  = len(x), len(delta)
    t     = np.arange(N)[:, None]              # (N, 1)
    logz  = (delta + 1j * 2 * np.pi * f)[None, :]  # (1, K)

    # V_N[t, k] = exp(t * log(z_k))  — no for-loop, outer product in log domain
    V_N   = np.exp(t * logz)                   # (N, K)

    # alpha = pinv(V_N) @ x
    alpha = np.linalg.pinv(V_N) @ x            # (K,)

    a   = np.abs(alpha)
    phi = np.angle(alpha)
    return a, phi

## 2.5 MUSIC Pseudo-Spectrum

In [ ]:
def MUSIC(x, n, K, n_f=200, n_delta=100,
          f_range=(0.0, 0.5), delta_range=(-0.1, 0.1), plot=True):
    """
    Compute and optionally plot the MUSIC 2-D pseudo-spectrum:
        P(z) = 1 / || W_perp^H  v^n(z) ||^2
    as a function of normalised frequency f and damping factor delta.

    Returns
    -------
    F_grid, D_grid : 2-D meshgrids
    log_P          : log10 of the pseudo-spectrum
    """
    N = len(x)
    l = N - n + 1
    _, W_perp = extract_signal_subspace(x, n, l, K)

    f_vals     = np.linspace(f_range[0],     f_range[1],     n_f)
    delta_vals = np.linspace(delta_range[0], delta_range[1], n_delta)
    F_grid, D_grid = np.meshgrid(f_vals, delta_vals)

    t = np.arange(n)[:, None]                  # (n, 1)
    log_P = np.zeros_like(F_grid)

    for i, d in enumerate(delta_vals):
        for j, fq in enumerate(f_vals):
            logz = d + 1j * 2 * np.pi * fq
            vn   = np.exp(t * logz).ravel()     # (n,)
            proj = W_perp.conj().T @ vn          # (n-K,)
            P    = 1.0 / (np.linalg.norm(proj)**2 + 1e-30)
            log_P[i, j] = np.log10(P)

    if plot:
        fig = plt.figure(figsize=(12, 6))
        ax  = fig.add_subplot(111, projection='3d')
        ax.plot_surface(F_grid, D_grid, log_P, cmap='viridis', linewidth=0)
        ax.set_xlabel('Frequency f')
        ax.set_ylabel('Damping δ')
        ax.set_zlabel('log10 P(z)')
        ax.set_title('MUSIC pseudo-spectrum')
        plt.tight_layout()
        plt.show()

    return F_grid, D_grid, log_P

---
# 3. Synthetic Signal — Section 3 of the Lab

## 3.1 Signal Synthesis


In [ ]:
np.random.seed(0)   # reproducibility

N  = 63
f0 = 0.25
f1 = f0 + 1.0 / N
a0, a1     = 1.0, 10.0
delta0, delta1 = 0.0, -0.05

# BUG FIX: original used np.random.uniform([-np.pi, np.pi]) which returns a
# single value from a (2,)-shaped array (wrong usage).  Correct call:
phi0, phi1 = np.random.uniform(-np.pi, np.pi, size=2)

# Build the synthetic signal
x_synth = Synthesis(
    N     = N,
    delta = np.array([delta0, delta1]),
    f     = np.array([f0,     f1]),
    a     = np.array([a0,     a1]),
    phi   = np.array([phi0,   phi1])
)

print(f'Synthetic signal: N={N}, f0={f0:.4f}, f1={f1:.4f}')
print(f'Phases drawn: phi0={phi0:.3f} rad, phi1={phi1:.3f} rad')

# Quick look
plt.figure(figsize=(10, 3))
plt.plot(np.real(x_synth), label='Re')
plt.plot(np.imag(x_synth), label='Im', alpha=0.6)
plt.xlabel('Sample index t'); plt.ylabel('Amplitude')
plt.title('Synthetic ESM signal')
plt.legend(); plt.tight_layout(); plt.show()

## 3.2 Spectral Analysis by Fourier Transform (Section 3.1)

We compare the periodogram **without** zero-padding ($N_{\rm fft}=N=63$) and **with** zero-padding ($N_{\rm fft}=1024$).

In [ ]:
def periodogram(signal, Nfft=None):
    """
    Compute the periodogram |X(f)|^2 / N.
    BUG FIX vs original: the original 'periodgramm' function computed the
    autocorrelation first and then its FFT — that gives a valid PSD estimate,
    but for a clean signal it is simpler and more direct to use |FFT(x)|^2.
    Also: the padding cell operated on `sum_padded` which included zeros that
    bias the autocorrelation estimate.  Here we zero-pad inside the FFT call.
    """
    N    = len(signal)
    if Nfft is None:
        Nfft = N
    X    = np.fft.fft(signal, n=Nfft)
    freq = np.fft.fftfreq(Nfft)
    psd  = np.abs(X)**2 / N
    return freq, psd


fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, Nfft, label in zip(
        axes,
        [N, 1024],
        ['No zero-padding  (Nfft = N = 63)', 'With zero-padding  (Nfft = 1024)']):
    freq, psd = periodogram(x_synth, Nfft)
    # Show only positive frequencies for readability
    idx = (freq >= 0)
    ax.semilogy(freq[idx], psd[idx])
    ax.axvline(f0, color='r', linestyle='--', label=f'f0={f0:.3f}')
    ax.axvline(f1, color='g', linestyle='--', label=f'f1={f1:.3f}')
    ax.set_xlabel('Normalised frequency')
    ax.set_ylabel('Power (log scale)')
    ax.set_title(label)
    ax.legend()

plt.tight_layout()
plt.show()

print(f"""
Comment:
  - Without zero-padding (Nfft=N=63) the two frequencies f0 and f1 are separated
    by exactly 1/N = the Fourier resolution limit, so they are NOT separable.
  - Zero-padding (Nfft=1024) interpolates the spectrum and reveals the two peaks,
    but does NOT improve the true spectral resolution: the two lines may still
    merge into one broad lobe (due to leakage) depending on their amplitude ratio.
  High-resolution methods (ESPRIT) overcome this limit.
""")

## 3.3 ESPRIT applied to the Synthetic Signal (Section 3.2.1)

In [ ]:
n_esprit = 32   # as recommended in the lab sheet
K        = 2

delta_hat, f_hat = ESPRIT(x_synth, n=n_esprit, K=K)

print('=== ESPRIT results on synthetic signal ===')
for k in range(K):
    print(f'  Pole {k}: delta={delta_hat[k]:+.5f}  f={f_hat[k]:+.6f}')

print('\nGround truth:')
print(f'  Pole 0: delta={delta0:+.5f}  f={f0:+.6f}')
print(f'  Pole 1: delta={delta1:+.5f}  f={f1:+.6f}')

print('\nErrors:')
# Match estimated poles to true poles by frequency
idx_sorted = np.argsort(f_hat)
f_sorted   = f_hat[idx_sorted]
d_sorted   = delta_hat[idx_sorted]
true_f     = [f0, f1]
true_d     = [delta0, delta1]
for k in range(K):
    print(f'  Pole {k}: Δf={f_sorted[k]-true_f[k]:+.2e}  Δdelta={d_sorted[k]-true_d[k]:+.2e}')

## 3.4 Least Squares — Amplitude and Phase Recovery (Section 3.2.1 step 4)

In [ ]:
a_hat, phi_hat = LeastSquares(x_synth, delta_hat, f_hat)

print('=== Least Squares amplitude/phase recovery ===')
# Sort by estimated frequency for comparison
for k in range(K):
    ki = idx_sorted[k]
    print(f'  Pole {k}: a={a_hat[ki]:.4f}  phi={phi_hat[ki]:.4f} rad')

print('\nGround truth:')
print(f'  Pole 0: a={a0:.4f}  phi={phi0:.4f} rad')
print(f'  Pole 1: a={a1:.4f}  phi={phi1:.4f} rad')

# Re-synthesise and compare
x_resynth = Synthesis(N, delta_hat, f_hat, a_hat, phi_hat)
err = np.linalg.norm(x_synth - x_resynth) / np.linalg.norm(x_synth)
print(f'\nRelative reconstruction error: {err:.2e}')

## 3.5 MUSIC Pseudo-Spectrum (Section 3.2.2)

In [ ]:
print('Computing MUSIC pseudo-spectrum (may take a few seconds)...')
F_grid, D_grid, log_P = MUSIC(
    x_synth, n=n_esprit, K=K,
    n_f=200, n_delta=100,
    f_range=(0.20, 0.30),
    delta_range=(-0.1, 0.1),
    plot=True
)
print('The two peaks should appear at (f0, delta0) and (f1, delta1).')

---
# 4. Audio Signals — Bell Sound Analysis (Section 4)



## 4.1 Load and visualise the bell sound

In [ ]:
# Try ClocheB first, fall back to ClocheA
import os
for fname in ['ClocheB.wav', 'ClocheB.WAV', 'ClocheA.wav', 'ClocheA.WAV']:
    if os.path.exists(fname):
        BELL_FILE = fname
        break
else:
    BELL_FILE = None
    print('WARNING: no bell WAV file found. '
          'Download ClocheA.wav or ClocheB.wav from eCampus and place '
          'it next to this notebook.')

if BELL_FILE:
    T_bell, data_bell, nframes_bell, nchannels_bell, Fs_bell = read_values(BELL_FILE)
    print(f'File         : {BELL_FILE}')
    print(f'Duration     : {T_bell:.3f} s')
    print(f'Sample rate  : {Fs_bell} Hz')
    print(f'Channels     : {nchannels_bell}')
    print(f'Frames       : {nframes_bell}')

    # Use mono (first channel)
    x_bell = data_bell[0].astype(float)
    t_bell = np.arange(len(x_bell)) / Fs_bell

    plot_sound(x_bell, t_bell, 0, T_bell, name=f'Bell sound — {BELL_FILE}', w=14, h=4)
    IPython.display.display(IPython.display.Audio(BELL_FILE))

## 4.2 Periodogram of the Bell Sound (Section 4.1)

In [ ]:
if BELL_FILE:
    Nfft_bell = 2**15   # large FFT for fine frequency resolution
    X_bell    = np.fft.rfft(x_bell, n=Nfft_bell)
    freq_bell = np.fft.rfftfreq(Nfft_bell, d=1.0/Fs_bell)  # in Hz
    psd_bell  = np.abs(X_bell)**2 / len(x_bell)

    plt.figure(figsize=(14, 4))
    plt.semilogy(freq_bell, psd_bell)
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Power (log scale)')
    plt.title(f'Periodogram of {BELL_FILE}')
    plt.xlim(0, Fs_bell / 2)
    plt.tight_layout()
    plt.show()

    # Identify dominant peaks and compare with the expected bell ratios
    # Expected ratios α_n = f_n / f_p:
    alpha_expected = [0.5, 1.0, 1.2, 1.5, 2.0, 2.5, 2.6, 2.7,
                      3.0, 3.3, 3.7, 4.2, 4.5, 5.0, 5.9]
    # Find approximate perceived pitch from the spectrum peak near fundamental
    # (rough estimate: peak in 200-800 Hz range)
    mask = (freq_bell > 200) & (freq_bell < 800)
    fp_est = freq_bell[mask][np.argmax(psd_bell[mask])]
    print(f'Estimated perceived pitch fp ≈ {fp_est:.1f} Hz')
    print(f'Expected eigenfrequencies (Hz):')
    for alpha in alpha_expected:
        print(f'  α={alpha:.1f}  →  f ≈ {alpha*fp_est:.1f} Hz')

## 4.3 ESPRIT Analysis of the Bell Sound (Section 4.2)

In [ ]:
if BELL_FILE:
    # Parameters from the lab sheet
    K_bell = 54
    n_bell = 512
    l_bell = 2 * n_bell          # = 1024
    N_bell = n_bell + l_bell - 1  # = 1535

    # Extract segment starting after the amplitude maximum (≥ 10000th sample)
    start  = 10000
    x_seg  = x_bell[start : start + N_bell]
    assert len(x_seg) == N_bell, 'Bell file is too short!'

    print(f'Analysing segment [{start}, {start+N_bell}] ({N_bell} samples)...')
    delta_bell, f_bell = ESPRIT(x_seg, n=n_bell, K=K_bell)

    # Convert normalised frequencies to Hz
    f_bell_Hz = f_bell * Fs_bell

    # Keep only physically meaningful poles: δ < 0 and f > 0
    mask_valid = (delta_bell < 0) & (f_bell_Hz > 0)
    delta_valid = delta_bell[mask_valid]
    f_valid     = f_bell[mask_valid]
    f_valid_Hz  = f_bell_Hz[mask_valid]

    # Sort by frequency for display
    sort_idx = np.argsort(f_valid_Hz)
    print(f'\nValid poles found: {np.sum(mask_valid)}')
    print(f'  {"k":>3}  {"f (Hz)":>10}  {"delta":>12}  {"T_decay (s)":>12}')
    for k in sort_idx:
        T_dec = -1.0 / (delta_valid[k] * Fs_bell)  # decay time constant in seconds
        print(f'  {k:>3}  {f_valid_Hz[k]:>10.2f}  {delta_valid[k]:>12.5f}  {T_dec:>12.3f}')

## 4.4 Least Squares — Bell Amplitudes and Phases

In [ ]:
if BELL_FILE:
    a_bell, phi_bell = LeastSquares(x_seg, delta_valid, f_valid)
    print('Amplitude / phase estimation done.')
    print(f'  Top 10 partials by amplitude:')
    amp_rank = np.argsort(a_bell)[::-1][:10]
    for r in amp_rank:
        print(f'    f={f_valid_Hz[r]:8.2f} Hz   a={a_bell[r]:.2e}   phi={phi_bell[r]:.3f} rad')

## 4.5 Re-synthesis and Listening (Section 4.2)

In [ ]:
if BELL_FILE:
    # Synthesise a longer signal to hear the resonances fade
    N_synth_bell = int(5 * Fs_bell)   # 5 seconds

    x_bell_resynth = Synthesis(
        N     = N_synth_bell,
        delta = delta_valid,
        f     = f_valid,
        a     = a_bell,
        phi   = phi_bell
    )

    # Take real part (physical signal) and normalise to int16 range
    x_rs_real  = np.real(x_bell_resynth)
    x_rs_norm  = (x_rs_real / np.max(np.abs(x_rs_real)) * 32000).astype(np.int16)

    out_file = 'bell_resynth.wav'
    write(out_file, Fs_bell, x_rs_norm)
    print(f'Re-synthesised bell saved to {out_file}')

    # Plot comparison
    t_rs = np.arange(N_synth_bell) / Fs_bell
    plt.figure(figsize=(14, 4))
    plt.plot(t_rs, x_rs_norm, linewidth=0.5)
    plt.xlabel('Time (s)'); plt.ylabel('Amplitude')
    plt.title('Re-synthesised bell sound (ESPRIT + Least Squares)')
    plt.tight_layout(); plt.show()

    IPython.display.display(IPython.display.Audio(out_file))

    print("""
Comment:
  The re-synthesised sound should closely match the original bell timbre.
  The ESPRIT method captures the exact frequencies and damping factors of each
  partial, including the non-harmonic ratios characteristic of bell acoustics
  (minor third, fifth, etc.). Because each partial is described individually,
  the synthesis can be extended to any duration while the resonances decay
  naturally — unlike Fourier-based synthesis which cannot model the
  frequency-dependent decay.
""")